# 01 · Cohort construction

Builds the analytic cohort from the NCDB liver Participant User File.

- **Input:** `data/raw/liver.csv` (NCDB PUF; path set in `hcc_tace.config`)
- **Output:** `data/processed/analysis_ready.csv`, plus the three advanced-stage pairwise
  cohorts `data/processed/df_{tace_sys,tace_same,same_sys}.csv` used for matching in `R/psm_matching.R`

### Load Data

In [1]:
import numpy as np
import pandas as pd

from hcc_tace import config
from hcc_tace import preprocessing as prep
from hcc_tace.cohort import advanced_stage, pairwise_cohorts

config.setup_notebook()

liver_data = pd.read_csv(config.RAW_NCDB_FILE)
liver_data.shape

(317111, 340)

### Include HCC

In [2]:
# 8171 (fibrolamellar) and 8180 (combined HCC-cholangiocarcinoma) are excluded
df = prep.select_hcc(liver_data)
df['HISTOLOGY'].value_counts()
df['PRIMARY_SITE'].value_counts()

PRIMARY_SITE
C220    284109
Name: count, dtype: int64

In [3]:
df.shape

(284109, 340)

### Include TACE and Systemic

In [4]:
df = prep.select_tace_and_systemic(df)
df.shape

(2418, 340)

### Exclude missing survival data

In [5]:
df = prep.select_known_survival(df)
df.shape

(2030, 340)

### Define time and event

In [6]:
df = prep.add_survival_outcome(df)  # event = 1 if dead

### Combine stage variables and exclude invalid stage data

In [7]:
df['ANALYTIC_STAGE_GROUP'].value_counts(), df['ANALYTIC_STAGE_GROUP'].isna().sum(), df['ANALYTIC_STAGE_GROUP'].dtype

(ANALYTIC_STAGE_GROUP
 4    752
 3    723
 2    265
 1    181
 9    108
 8      1
 Name: count, dtype: int64,
 np.int64(0),
 dtype('int64'))

In [8]:
df = prep.add_stage(df)
df['COMBINED_STAGE_GROUP'].value_counts()

COMBINED_STAGE_GROUP
4    752
3    723
2    265
1    181
Name: count, dtype: int64

In [9]:
df['COMBINED_STAGE_GROUP'] = df['COMBINED_STAGE_GROUP'].astype(int)
df.shape

(1921, 343)

### Filter Year of Diagnosis

In [10]:
df['YEAR_OF_DIAGNOSIS'].describe()

count    1921.000000
mean     2018.127538
std         3.446296
min      2004.000000
25%      2018.000000
50%      2019.000000
75%      2020.000000
max      2021.000000
Name: YEAR_OF_DIAGNOSIS, dtype: float64

In [11]:
df = df[df['YEAR_OF_DIAGNOSIS'] >= 2007]
df.shape

(1858, 343)

### Define sequence label

In [12]:
# 0 = TACE first, 1 = systemic first, 2 = simultaneous (within 3 days)
df = prep.add_sequence_group(df)
df['sequence_group'].value_counts()

sequence_group
0    1212
2     362
1     284
Name: count, dtype: int64

### Define surgery

In [13]:
df = prep.add_surgery_flags(df)
df['Surgery'].value_counts()

Surgery
No     1668
Yes     190
Name: count, dtype: int64

In [14]:
df['DX_DEFSURG_STARTED_DAYS'].describe()

count     189.000000
mean      165.465608
std       172.740012
min         0.000000
25%        44.000000
50%       119.000000
75%       216.000000
max      1141.000000
Name: DX_DEFSURG_STARTED_DAYS, dtype: float64

In [15]:
df[['Prior_Surgery', 'Subsequent_Surgery', 'Surgery_Between']].apply(pd.Series.value_counts)

,Prior_Surgery,Subsequent_Surgery,Surgery_Between
No,1804,1773,1808
Yes,54,85,50


### Recode other covariates

In [16]:
df = prep.recode_covariates(df)

Missingness across the full candidate covariate set:

In [17]:
#Updated covariates list with new names
covariates = [
    'AGE', 'SEX', 'RACE', 'HISPANIC', #Demographics
    'INSURANCE_STATUS', 'MEDIAN_INCOME', 'NoHighSchooldegree2016_2020',#Socioeconomic
    'Charlson_Deyo_Score', 'HISTOLOGY', 'TUMOR_SIZE', 'COMBINED_STAGE_GROUP', 'COMBINED_CLIN_T', 'Radiation_Therapy', #Clinical
    'FACILITY_TYPE', 'FACILITY_REGION', #Facility
    'YEAR_OF_DIAGNOSIS', #Temporal
    'Subsequent_Surgery'  # Surgery status
]

In [18]:
df[covariates].isna().sum()

AGE                              0
SEX                              0
RACE                             0
HISPANIC                        29
INSURANCE_STATUS                 0
MEDIAN_INCOME                  357
NoHighSchooldegree2016_2020    354
Charlson_Deyo_Score              0
HISTOLOGY                        0
TUMOR_SIZE                     400
COMBINED_STAGE_GROUP             0
COMBINED_CLIN_T                 94
Radiation_Therapy                0
FACILITY_TYPE                   45
FACILITY_REGION                 45
YEAR_OF_DIAGNOSIS                0
Subsequent_Surgery               0
dtype: int64

### Final covariates list

In [19]:
covariates = config.COVARIATES
print(df['INSURANCE_STATUS'].isna().sum())
print(df['Radiation_Therapy'].isna().sum())

df = df.dropna(subset=covariates)
df['sequence_group'].value_counts()

0
0


sequence_group
0    1212
2     362
1     284
Name: count, dtype: int64

In [20]:
df.shape

(1858, 357)

### Save

In [21]:
df.to_csv(config.ANALYSIS_READY, index=False)

### Pairwise cohorts for propensity score matching

Advanced stage (III-IV) only. Each pair is recoded so the first group is 0 and the second is 1.

In [22]:
df = pd.read_csv(config.ANALYSIS_READY)
for name, df_pair in pairwise_cohorts(advanced_stage(df)).items():
    df_pair.to_csv(config.PROCESSED_DIR / f'df_{name}.csv', index=False)
    print(name, df_pair['sequence_group'].value_counts().to_dict())

tace_sys {0: 930, 1: 205}
tace_same {0: 930, 1: 291}
same_sys {0: 291, 1: 205}
